# GLM intro

Generalised Linear Models (GLMs) are simply a class of models that use linear predictors to make predictions. Infact, the linear regression we've been using this whole time is just one subset model from this class that uses a specific "link" function. 

The generalised part of linear models relates to the notion of this "link" function. There are many link functions available such as ... 

## linear regression as a GLM

info ...


## Probit

info ...


## Logistic Regression


info ...



## GLM case study: Are Emily and Greg More Employable Than Lakisha and Jamal?
To explore the implementation of GLMs (specifically probit and logit), we will look at a famous "audit" study from 2004 by Marianne Bertrand and Snedhil Mullainthan titled: ""


To summarise quickly ... 

Here is the paper link: https://www.povertyactionlab.org/sites/default/files/research-paper/3%20A%20Field%20Experiment%20on%20Labor%20Market%20Discrimination%20Sep%2004.pdf.

I pulled the data from Open ICPSR: https://www.openicpsr.org/openicpsr/project/116023/version/V1/view?path=/openicpsr/116023/fcr:versions/V1/lakisha_aer.dta&type=file


The table below details all the variables we will utilise today *(there are a lot more variables available, but we don't need them for our purposes today)*: 

'const', 
     "white_dummy",
     "h",
     'city_dummy',
     'sex_dummy',
     'honors',
     'volunteer',
     'military', 
     'email',
     'empholes',
     'workinschool', 
     'computerskills', 
     'specialskills'

| Variable | Description |
| :--- | :--- |
| `yearsexp` | Number of years of work experience on the resume |
| `honors` | Indicator: 1 if resume mentions honors |
| `volunteer` | Indicator: 1 if resume mentions volunteering experience |
| `military` | Indicator: 1 if resume mentions military experience |
| `empholes` | Indicator: 1 if resume mentions employment holes |
| `workinschool` | Indicator: 1 if resume mentions work experience while at school |
| `email` | Indicator: 1 if email address is on the resume |
| `computerskills` | Indicator: 1 if resume mentions computer skills |
| `specialskills` | Indicator: 1 if resume mentions special skills |
| `sex` | Sex ('f' = female; 'm' = male) |
| `race` | Race ('b' = black; 'w' = white) |
| `h` | Indicator: 1 = high quality resume |
| `l` | Indicator: 1 = low quality resume |
| `call` | Indicator: 1 = applicant was called back |
| `city` | City ('c' = Chicago; 'b' = Boston) |
| `ad_id` | Employment ad identifier |

Now, with all that out of the way let's code this up!

Firstly, clear the memory if you have anything else loaded in:

In [35]:
%reset -f

Firstly, as per usual, we load in the necessary packages and data:

In [36]:
import numpy             as np
import statsmodels.api   as sm
import pandas            as pd
import seaborn           as sns

from statsmodels.discrete.discrete_model import LogitResults
from scipy.stats import ttest_ind
from statsmodels.iolib.summary2 import summary_col

In [37]:
df = pd.read_stata("https://raw.githubusercontent.com/Michael-Morgan-Giles/regression_analysis/refs/heads/main/data/lakisha_aer.dta")

We then create some helpful columns for our modeling:

In [38]:
df = sm.add_constant(df)
df['city_dummy'] = np.where(df["city"] == "c",1,0)
df['white_dummy'] = np.where(df['race']=="w", 1,0)
df['sex_dummy'] = np.where(df["sex"] == "m",1,0)
df["hl_dummy"] = np.where(df['h'] == 1, "high", "low")
df.columns

Index(['const', 'id', 'ad', 'education', 'ofjobs', 'yearsexp', 'honors',
       'volunteer', 'military', 'empholes', 'occupspecific', 'occupbroad',
       'workinschool', 'email', 'computerskills', 'specialskills', 'firstname',
       'sex', 'race', 'h', 'l', 'call', 'city', 'kind', 'adid', 'fracblack',
       'fracwhite', 'lmedhhinc', 'fracdropout', 'fraccolp', 'linc', 'col',
       'expminreq', 'schoolreq', 'eoe', 'parent_sales', 'parent_emp',
       'branch_sales', 'branch_emp', 'fed', 'fracblack_empzip',
       'fracwhite_empzip', 'lmedhhinc_empzip', 'fracdropout_empzip',
       'fraccolp_empzip', 'linc_empzip', 'manager', 'supervisor', 'secretary',
       'offsupport', 'salesrep', 'retailsales', 'req', 'expreq', 'comreq',
       'educreq', 'compreq', 'orgreq', 'manuf', 'transcom', 'bankreal',
       'trade', 'busservice', 'othservice', 'missind', 'ownership',
       'city_dummy', 'white_dummy', 'sex_dummy', 'hl_dummy'],
      dtype='object')

Next, we set up a list contianing all the varibales in `X` *(except for one continuous variable we will add in later to make estimating marginal effects easier)*.

We will also label our outcome variable and the column to filter for African-American and White names. 

In [39]:
# Set up lists to filter data in modelling stage
X = ['const', 
     "white_dummy",
     "h",
     'city_dummy',
     'sex_dummy',
     'honors',
     'volunteer',
     'military', 
     'email',
     'empholes',
     'workinschool', 
     'computerskills', 
     'specialskills'
    ]

y = "call" # dependent variable is the callback dumm
# set up objects to filter for African-American and white names
T = 'race'

Firstly, let's recreate some of the key initial findings of the paper, namely the average callback rates by name group:

In [40]:
df.groupby(T)[y].describe()

,count,mean,std,min,25%,50%,75%,max
race,,,,,,,,
b,2435.0,0.064476,0.245649,0.0,0.0,0.0,0.0,1.0
w,2435.0,0.096509,0.295346,0.0,0.0,0.0,0.0,1.0


As, we can see in the table above, the mean callback rate for black sounding names is ~6.5% whilst the mean callback rate for white sounding names is ~ 9.5%, a ~3% higher callback rate on average. 

That might not seem like a large difference, however, given that 


All this equates to the inferrence that *"... , these results imply that a White applicant should expect on average one callback for every 10 ads she or he applies to; on the other  hand, an African-American applicant would need to apply to about 15 different ads to achieve the same result"* (pg. 998).

We can also show the benefit white names recieved in terms of callback rates when considering the "quality" of the resume. This can be easily achieved by grouping by the `hl_dummy` variable as such:

In [41]:
pd.pivot_table(df.groupby([T,"hl_dummy"])[y].mean().reset_index(), 
               index = "race", 
               columns = "hl_dummy", 
               values = 'call').assign(delta=lambda x: x['high'] - x['low']).reset_index()

hl_dummy,race,high,low,delta
0,b,0.067048,0.061881,0.005167
1,w,0.107931,0.084983,0.022948


We see that there is an ~2 percentage point increase in callback rates how white sounding names who have a higher quality resume *(an approx ~25% increase in callback rate)*. There is no similar increase for high quality resumes with a black sounding name.  

... 

Now we have replicated the main results of the paper, but say we wanted to account for varaibles that might be confounding ...

Before we get to the GLM modeling, we might want to investigate how well the randomisation process worked when assigning names to resumes. 

Because the names were randomly assigned based on how "African American" or "White" the name sounds, we should expect that all the other variables we observe converge to the same mean, such that the only detectable difference between the resumes is the sound of the name. 

We can investigate this by comparing the means of all our covariates in `X` (plus `yearsexp`) between the Black and White sounding names:

In [42]:
df.groupby(T)[X[1:] + ["yearsexp"]].mean().T.assign(delta=lambda x: x['b'] - x['w']).reset_index()


race,index,b,w,delta
0,white_dummy,0.000000,1.000000,-1.000000
1,h,0.502259,0.502259,0.000000
2,city_dummy,0.555236,0.555236,0.000000
3,sex_dummy,0.225462,0.236140,-0.010678
4,honors,0.051335,0.054209,-0.002875
5,volunteer,0.414374,0.408624,0.005749
6,military,0.101848,0.092402,0.009446
7,email,0.479671,0.478850,0.000821
8,empholes,0.445996,0.450103,-0.004107
9,workinschool,0.560986,0.558111,0.002875


As we can see from the table above, the only varibales with a noticable difference is the `white_dummy` variable (which makes sense considering it is mutually exclusive). 

This is really nice evidence that any differences we observe between the two groups can be attributed largely to the difference in perceptions of the names and not any of these other factors i.e. the difference in ballback rates can be attributed to the different sounding names. 

We can take this analysis one step further by calculating a mean group $t$-test to each variable above to determine if a statistically significant difference in the means can be detected for each variable. 

We can do that by using the `ttest_ind` function from `scipy` as such:

In [43]:
# set up dataframe to store results
t_results = pd.DataFrame({"index" : [],
                          "t-stat" : [], 
                          "p-value" : []})

# for loop over every column we want to measure
for c in X[1:] + ["yearsexp"]:
    t, p = ttest_ind(df.loc[df[T] == "b",c], df.loc[df[T] == "w",c])
    
    t_results = pd.concat([t_results,
                          pd.DataFrame({"index" : [c],
                                        "t-stat" : [t], 
                                        "p-value" : [p]})])
    
    del t, p 

pd.merge(df.groupby(T)[X[1:] + ["yearsexp"]].mean().T.assign(delta=lambda x: x['b'] - x['w']).reset_index(),
         t_results,
         left_on = "index",
         right_on = "index", 
         how = "left")

/usr/lib/python3/dist-packages/scipy/stats/_axis_nan_policy.py:523: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


,index,b,w,delta,t-stat,p-value
0,white_dummy,0.000000,1.000000,-1.000000,-inf,0.000000
1,h,0.502259,0.502259,0.000000,0.000000,1.000000
2,city_dummy,0.555236,0.555236,0.000000,0.000000,1.000000
3,sex_dummy,0.225462,0.236140,-0.010678,-0.884131,0.376669
4,honors,0.051335,0.054209,-0.002875,-0.448564,0.653766
5,volunteer,0.414374,0.408624,0.005749,0.407590,0.683592
6,military,0.101848,0.092402,0.009446,1.112883,0.265814
7,email,0.479671,0.478850,0.000821,0.057356,0.954264
8,empholes,0.445996,0.450103,-0.004107,-0.288096,0.773286
9,workinschool,0.560986,0.558111,0.002875,0.202013,0.839915


In [44]:
del t_results

Now one difference we might expected is between high and low "quality" resumes. For example, a high quality resume is by definition one that might have more education such as an honors degree, might volunteer, maybe they have a special skill or have more years of experience. 

In [45]:
# set up dataframe to store results
t_results_quality = pd.DataFrame({"index" : [],
                          "t-stat" : [], 
                          "p-value" : []})

# for loop over every column we want to measure
for c in X[1:] + ["yearsexp"]:
    t, p = ttest_ind(df.loc[df["hl_dummy"] == "high",c], df.loc[df["hl_dummy"] == "low",c])
    
    t_results_quality = pd.concat([t_results_quality,
                          pd.DataFrame({"index" : [c],
                                        "t-stat" : [t], 
                                        "p-value" : [p]})])
    
    del t, p 

# merge together mean effects grouped by quality dummy variable and calculate different between high and low, 
# with the t results table
pd.merge(df.groupby("hl_dummy")[X[1:] + ["yearsexp"]].mean().T.assign(delta=lambda x: x['high'] - x['low']).reset_index(),
         t_results_quality,
         left_on = "index",
         right_on = "index", 
         how = "left")

/usr/lib/python3/dist-packages/scipy/stats/_axis_nan_policy.py:523: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


,index,high,low,delta,t-stat,p-value
0,white_dummy,0.500000,0.500000,0.000000,0.000000,1.000000e+00
1,h,1.000000,0.000000,1.000000,inf,0.000000e+00
2,city_dummy,0.557645,0.552805,0.004840,0.339762,7.340505e-01
3,sex_dummy,0.231807,0.229785,0.002022,0.167374,8.670824e-01
4,honors,0.071137,0.034241,0.036896,5.776567,8.097871e-09
5,volunteer,0.792314,0.027228,0.765086,86.217922,0.000000e+00
6,military,0.190106,0.003300,0.186806,23.190282,6.177185e-113
7,email,0.923957,0.030528,0.893429,139.357264,0.000000e+00
8,empholes,0.339738,0.557343,-0.217605,-15.644000,7.340338e-54
9,workinschool,0.718316,0.399340,0.318976,23.669248,2.529881e-117


Now as we can see, most of the covariates show statistically significant differences between high and low "quality" resumes. This is generally as expected, yet also provides evidence that those with higher "quality" resumes might expect more callbacks generally (independent of the sound of their name). 

Now we're ready to test what the difference in call back rates between different "sounding" names and different "quality" of resume are using GLMs.

# Probit coefficients and Marginal effect table

Firstly, we will use a probit model to estimate the effect on call back rates depending on the if the resume name sounds White or African America. Our coefficient of interest will be on the `white_dummy` variable.

Handly for us `statsmodels` have a function for Probit so we don't have to directly code up the MLE solution ourselves. Instead we can call the `Probit()` function below as such *(note I am clustering the standard errors at the ad level as per the paper)*:

In [46]:
probit_res = sm.Probit(df[y], 
                       df[X + ['yearsexp'] ]).fit(cov_type='cluster',
                                                         cov_kwds={'groups': df['adid']})

print(probit_res.summary())

Optimization terminated successfully.
         Current function value: 0.266630
         Iterations 6
                          Probit Regression Results                           
Dep. Variable:                   call   No. Observations:                 4870
Model:                         Probit   Df Residuals:                     4856
Method:                           MLE   Df Model:                           13
Date:                Wed, 01 Jul 2026   Pseudo R-squ.:                 0.04765
Time:                        16:56:53   Log-Likelihood:                -1298.5
converged:                       True   LL-Null:                       -1363.5
Covariance Type:              cluster   LLR p-value:                 2.144e-21
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const             -1.7383      0.121    -14.335      0.000      -1.976      -1.501
white_dummy      

Now we can see that there is a positive and statistically significant coefficient for the `white_dummy` variables. However, as mentioned above, it is difficult to interpret what the magnitude of these coefficients translates into after going through their activation function. 

So, we have to estimate the marginal effects for each of these coefficients to be able to more easily interpret the results. Luckily for us, `statsmodels` also has a handy `get_margeff()` function that calculates the marginal effects for a discrete choice model such as Probit. 

For info on marginal effects calcs see: https://www.statsmodels.org/stable/generated/statsmodels.discrete.discrete_model.DiscreteResults.get_margeff.html. 

For now, we want to estimate the derivative of each observation whilst accounting the the fact that most of our variables are dummy variables (hence the `dummy` arguement having a list of the columns to treat as dummy variables):

In [47]:
probit_me = probit_res.get_margeff(at = "overall",
                                   method = "dydx",
                                   dummy = X)
print(probit_me.summary())

       Probit Marginal Effects       
Dep. Variable:                   call
Method:                          dydx
At:                           overall
                    dy/dx    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
white_dummy        0.0304      0.006      4.945      0.000       0.018       0.042
h                  0.0197      0.018      1.086      0.277      -0.016       0.055
city_dummy        -0.0295      0.012     -2.434      0.015      -0.053      -0.006
sex_dummy         -0.0034      0.011     -0.296      0.767      -0.026       0.019
honors             0.0440      0.021      2.101      0.036       0.003       0.085
volunteer         -0.0143      0.012     -1.152      0.249      -0.039       0.010
military          -0.0068      0.013     -0.504      0.614      -0.033       0.020
email              0.0066      0.015      0.435      0.663      -0.023       0.036
empholes          

Okay, now the marginal effect for `white_dummy` is $0.03$. In words, that means a resume with a "white" sounding name recieves 3% higher call back rates, cetirus paribus. 

Okay, great, but what does that mean in context. Well, if we consider that the baseline callback rate for job applications was ~ $8\%$ for all resumes *(black and white sounding names together)*; a 3% increase in the likelihood of recieving a callback simply for having a white sounding name actually equates to an ~ 38% relative increase in callback rate. The maths is below: 

$$ baseline \ Callbacks + Callbacks \ from \ White \ Name = 8\% + 3\% = 11\% $$ 

However, a 3% point increase relative to a 8% baseline callback rate equates to: 

$$ \frac{3\%}{8\%} \approx 38\% $$

That is, having a white name on your resume means you have a 38% increase over the baseline callback rate:

In [48]:
probit_me.margeff[0] / df[y].mean() 

0.37724929319921413

This is quite a large effect. Indeed, it is also nearly 50% of the likelihood of receiving a callback if the resume has a black sounding name (6%):

In [49]:
probit_me.margeff[0] / df.loc[df[T] == "b",y].mean() 

0.47096092522067934

All this equates to the inferrence that *"... , these results imply that a White applicant should expect on average one callback for every 10 ads she or he applies to; on the other  hand, an African-American applicant would need to apply to about 15 different ads to achieve the same result"* (pg. 998).

# Logit coefficients and Marginal effect table

Okay, now that we've had a look at how to estimate the Probit model, let's do the same with a logistic regression. 

In terms of coding this up it's functionally identical, however instead of the `Probit()` function from `statsmodels`, we will use the `Logit()` model as such: 

In [50]:
logit_res = sm.Logit(df[y], 
                     df[X + ['yearsexp'] ]).fit(cov_type='cluster', 
                                         cov_kwds={'groups': df['adid']})
print(logit_res.summary())

Optimization terminated successfully.
         Current function value: 0.266567
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:                   call   No. Observations:                 4870
Model:                          Logit   Df Residuals:                     4856
Method:                           MLE   Df Model:                           13
Date:                Wed, 01 Jul 2026   Pseudo R-squ.:                 0.04788
Time:                        16:56:53   Log-Likelihood:                -1298.2
converged:                       True   LL-Null:                       -1363.5
Covariance Type:              cluster   LLR p-value:                 1.618e-21
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const             -3.1044      0.241    -12.866      0.000      -3.577      -2.631
white_dummy      

We can see that the coefficients now look a bit different than the Probit results. However, we can't easily interpret these without finding the marginal effects. Again we can use the `get_margeff()` function to estimate these from the Logitistic model as such:

In [51]:
logit_me = logit_res.get_margeff(at = "overall",
                                   method = "dydx",
                                   dummy = X)
print(logit_me.summary())

        Logit Marginal Effects       
Dep. Variable:                   call
Method:                          dydx
At:                           overall
                    dy/dx    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
white_dummy        0.0313      0.006      5.038      0.000       0.019       0.043
h                  0.0203      0.018      1.153      0.249      -0.014       0.055
city_dummy        -0.0299      0.012     -2.446      0.014      -0.054      -0.006
sex_dummy         -0.0031      0.012     -0.263      0.792      -0.026       0.020
honors             0.0401      0.021      1.934      0.053      -0.001       0.081
volunteer         -0.0144      0.012     -1.183      0.237      -0.038       0.009
military          -0.0071      0.014     -0.520      0.603      -0.034       0.020
email              0.0076      0.014      0.530      0.596      -0.021       0.036
empholes          

As  we can see, now the marginal effects are very similar to the Probit effects. This means we can an overall similar relative increase in callbacks for white sounding names of ~38%.

In [52]:
logit_me.margeff[0] / df[y].mean() 

0.3884449377895282

In [53]:
probit_me.margeff[0] / df.loc[df[T] == "b",y].mean() 

0.47096092522067934

The really solid thing about this research isn't the modelling. It is the research design that used randomisation to isolate the effect to the inference behind the culturally latent information in names. 

This allows the researchers to isolate a large sample size to estimate an effect on the racialised dimensions of the united states labour market. The models are just a tool that can be used to remain rigurous in your measurements.